# QLoRA Transaction Classifier (Qwen2.5-1.5B-Instruct)

Phase 3-4 of the Spend Insights fine-tuning project. Trains a LoRA adapter on top of
Qwen2.5-1.5B-Instruct in 4-bit (bitsandbytes NF4) on a Colab T4.

**Before running:** build the dataset locally with
`python train/build_dataset.py` and have `data/train.jsonl`, `data/val.jsonl`,
`data/test.jsonl` ready to upload.

In [ ]:
import sys, torch

print("Python:", sys.version.split()[0])
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run.")


In [ ]:
!pip install -q --upgrade transformers peft "trl>=0.12,<0.13" accelerate bitsandbytes datasets sentencepiece einops scikit-learn

import transformers, trl
print("transformers", transformers.__version__, "| trl", trl.__version__)


In [ ]:
# Upload data/train.jsonl, data/val.jsonl, data/test.jsonl (built locally).
from google.colab import files
import json

uploaded = files.upload()

def _rows(name):
    assert name in uploaded, f"Missing {name} - upload all three .jsonl files from data/"
    return [json.loads(l) for l in uploaded[name].decode("utf-8").splitlines() if l.strip()]

TRAIN_ROWS = _rows("train.jsonl")
VAL_ROWS = _rows("val.jsonl")
print(f"Loaded {len(TRAIN_ROWS)} train / {len(VAL_ROWS)} val examples")


In [ ]:
import torch
from datasets import Dataset

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "outputs/qlora-transaction-classifier"

CATEGORIES = [
    "Food & Dining", "Groceries", "Transport", "Shopping", "Subscriptions",
    "Bills & Utilities", "Entertainment", "Healthcare", "Transfers", "Other",
]

SYSTEM_PROMPT = (
    "You are a bank transaction categorizer for a personal finance app. "
    "Categorize each transaction into exactly one of these categories: {cats}. "
    "Respond with only the category name."
)

CONFIG = {
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "target_modules": ["q_proj", "v_proj"]},
    "train": {
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "num_train_epochs": 4,
        "max_seq_length": 512,
        "warmup_ratio": 0.1,
        "logging_steps": 10,
        "eval_steps": 50,
        "save_steps": 250,
        "save_total_limit": 2,
        "bf16": True,
        "packing": False,
    },
}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

model = get_peft_model(model, LoraConfig(
    r=CONFIG["lora"]["r"],
    lora_alpha=CONFIG["lora"]["alpha"],
    lora_dropout=CONFIG["lora"]["dropout"],
    target_modules=CONFIG["lora"]["target_modules"],
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()


In [ ]:
def to_text(ex):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(cats=", ".join(CATEGORIES))},
        {"role": "user", "content": f"Categorize this transaction: {ex['input']}"},
        {"role": "assistant", "content": ex["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list(TRAIN_ROWS).map(lambda ex: {"text": to_text(ex)})
val_ds = Dataset.from_list(VAL_ROWS).map(lambda ex: {"text": to_text(ex)})
print("Sample formatted example (tail):
", train_ds[0]["text"][-400:])


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

t = CONFIG["train"]
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=t["max_seq_length"],
    packing=t["packing"],
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=t["learning_rate"],
        per_device_train_batch_size=t["per_device_train_batch_size"],
        gradient_accumulation_steps=t["gradient_accumulation_steps"],
        num_train_epochs=t["num_train_epochs"],
        warmup_ratio=t["warmup_ratio"],
        lr_scheduler_type="cosine",
        logging_steps=t["logging_steps"],
        eval_strategy="steps",
        eval_steps=t["eval_steps"],
        save_strategy="steps",
        save_steps=t["save_steps"],
        save_total_limit=t["save_total_limit"],
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        gradient_checkpointing=True,
        bf16=t["bf16"],
        fp16=False,
        report_to="none",
        seed=42,
    ),
)

trainer.train()
print("Training done. Watch for eval_loss decreasing - if flat, debug the data format first.")


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

import shutil
from google.colab import files

shutil.make_archive("adapter", "zip", OUTPUT_DIR)
files.download("adapter.zip")
print("Downloaded adapter.zip - unzip into spend-classifier-qlora/outputs/ and run eval/eval_finetuned.py")


In [ ]:
def categorize(ex):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(cats=", ".join(CATEGORIES))},
        {"role": "user", "content": f"Categorize this transaction: {ex['input']}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

import random
random.seed(1)
for ex in random.sample(VAL_ROWS, 10):
    print(f"{categorize(ex):18} | gold: {ex['output']}")


## Next steps

1. Download the adapter zip (cell above), unzip into `spend-classifier-qlora/outputs/qlora-transaction-classifier`.
2. Locally: `python eval/eval_finetuned.py --adapter outputs/qlora-transaction-classifier` and compare against
   `eval/results/baseline_rules.csv` / `baseline_groq.csv`.
3. Integrate: set `CATEGORIZER_MODE=local` and use `inference/classifier.py`.
